In [ ]:
# =============================================================================
# UK ENERGY SECTOR RAG SYSTEM — NOTEBOOK GUIDE
# =============================================================================
#
# ABOUT
# -----
# This notebook implements a multi-technique Retrieval-Augmented Generation
# (RAG) system for the UK energy sector, built on public documents from NESO
# and Ofgem. It covers the full pipeline: document ingestion, chunking, hybrid
# BM25-FAISS retrieval, cross-encoder reranking, HyDE-based query enhancement,
# LLM generation, and evaluation against a baseline.
#
# HOW TO USE
# ----------
# 1. ADD DATA FILES
#    Place the provided data files in the following directory:
#
#       /content/sample_data/data/
#
#    Expected files:
#       - NESO_Winter_Outlook_2025_26.pdf
#       - ETYS_2024.pdf
#       - Ofgem_State_of_the_Market_2024.pdf
#       - FES_2025_Executive_Summary.pdf
#       - NESO_Winter_Outlook_Data_Workbook.xlsx
#
# 2. USING A DIFFERENT STORAGE LOCATION?
#    If your files are stored elsewhere, update the file paths in:
#       - Cell 2  → PDF ingestion paths
#       - Cell 5  → Excel workbook ingestion path
#
# 3. RUN CELLS IN ORDER
#    Execute cells sequentially from top to bottom. Each cell builds on
#    the outputs of the previous one.
#
# 4. API KEY
#    Ensure your Anthropic API key is set before running generation cells.
#    You can add it via Colab Secrets or directly in the credentials cell.
#
# =============================================================================

In [12]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================
# pdfplumber        : extracts text from PDFs (handles tables well)
# sentence-transformers : embeddings (Step 1c) + cross-encoder reranker (Step 3)
# faiss-cpu         : vector store for Step 1d (CPU version, fine for Colab)
# rank_bm25         : keyword retrieval for Step 2
# openpyxl          : reads Excel workbook for Step 1a data ingestion

!pip install pdfplumber sentence-transformers faiss-cpu rank_bm25 openpyxl -q


In [13]:
# ============================================================
# CELL 2 — Step 1a: PDF Parsing + Text Cleaning
# ============================================================
# Two sub-steps combined here:
#   (i)  extract_text_from_pdf  : pulls raw text from each PDF page by page
#   (ii) clean_text             : strips PDF artefacts before chunking
#                                 (page markers, truncated words, headers)

import pdfplumber
import os
import re

# --- Define where your PDFs live ---
PDF_FOLDER = '/content/sample_data/data/'


def extract_text_from_pdf(pdf_path):
    """
    Opens a PDF and extracts all text page by page.
    Returns one big string with all the text from the document.
    """
    full_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            text = page.extract_text()  # extracts text from one page
            if text:  # some pages are images/blank — skip those
                full_text += f"\n--- Page {page_num + 1} ---\n"
                full_text += text
    return full_text


def clean_text(text):
    """
    Removes PDF extraction artefacts that pollute chunks and confuse retrieval:
      - Page markers:  --- Page 14 ---
      - Section headers in path format: 14/Winter Outlook 2025-26/Demand
      - Standalone page numbers floating alone on a line
      - Excessive consecutive newlines
      - Multiple spaces
      - Very short lines under 4 chars (usually stray numbers or symbols)
    """
    # Remove page markers like '--- Page 14 ---'
    text = re.sub(r'---\s*Page\s*\d+\s*---', ' ', text)

    # Remove path-style section headers like '14/Winter Outlook 2025-26/Demand and Supply'
    text = re.sub(r'\d+/[\w\s\-\u2013]+/[\w\s\-\u2013]+\n', ' ', text)

    # Remove standalone numbers (floating page numbers)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    # Remove standalone Figure header lines e.g. 'Figure 6:'
    text = re.sub(r'^Figure\s+\d+[:\-]\s*$', '', text, flags=re.MULTILINE)

    # Collapse 3+ consecutive newlines into 2
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Collapse multiple spaces into one
    text = re.sub(r'  +', ' ', text)

    # Strip each line; drop lines under 4 chars (artefacts)
    lines = [line.strip() for line in text.split('\n')]
    lines = [line for line in lines if len(line) >= 4 or line == '']

    return '\n'.join(lines).strip()


# --- Loop through all PDFs, extract and clean ---
documents = {}  # dictionary: {filename: cleaned_text}

for filename in os.listdir(PDF_FOLDER):
    if filename.endswith('.pdf'):
        filepath = os.path.join(PDF_FOLDER, filename)
        print(f"Extracting: {filename}...")
        raw_text     = extract_text_from_pdf(filepath)
        cleaned_text = clean_text(raw_text)
        documents[filename] = cleaned_text

# --- Sanity check ---
print(f"\n Extracted and cleaned {len(documents)} documents")
for name, text in documents.items():
    print(f"  {name}: {len(text):,} characters")


Extracting: ETYS24 Publication.pdf...
Extracting: neso-winter-outlook-2025-26.pdf...
Extracting: future-energy-scenarios-2025-executive-summary_0.pdf...
Extracting: OFG2296_State of the Market Report.pdf...

 Extracted and cleaned 4 documents
  ETYS24 Publication.pdf: 95,254 characters
  neso-winter-outlook-2025-26.pdf: 56,070 characters
  future-energy-scenarios-2025-executive-summary_0.pdf: 41,606 characters
  OFG2296_State of the Market Report.pdf: 24,858 characters


In [14]:
# ============================================================
# CELL 3 — Step 1b: Chunking
# ============================================================

# We use a recursive character splitter — it tries to split on
# paragraph breaks (\n\n) first, then sentences (\n), then words.
# This is smarter than splitting every N characters blindly.

# No need to install anything new — we use a lightweight approach
# without importing all of LangChain

def recursive_chunk(text, chunk_size=800, chunk_overlap=150):
    """
    Splits a long text into overlapping chunks.
    Tries to split on natural boundaries (paragraphs, newlines)
    rather than cutting mid-sentence.

    Returns: list of text strings (the chunks)
    """
    # Natural split points — try these in order of preference
    separators = ["\n\n", "\n", ". ", " "]

    chunks = []

    # If text is already small enough, return as-is
    if len(text) <= chunk_size:
        return [text.strip()]

    # Try each separator until we find one that works
    for sep in separators:
        if sep in text:
            parts = text.split(sep)
            current_chunk = ""

            for part in parts:
                # If adding this part keeps us under chunk_size, add it
                if len(current_chunk) + len(part) + len(sep) <= chunk_size:
                    current_chunk += part + sep
                else:
                    # Save current chunk if it has content
                    if current_chunk.strip():
                        chunks.append(current_chunk.strip())

                    # Start new chunk — include overlap from previous chunk
                    if chunks:
                        # Take last `chunk_overlap` characters as overlap
                        overlap_text = chunks[-1][-chunk_overlap:]
                        current_chunk = overlap_text + sep + part + sep
                    else:
                        current_chunk = part + sep

            # Don't forget the last chunk
            if current_chunk.strip():
                chunks.append(current_chunk.strip())

            if chunks:  # if this separator worked, stop trying others
                break

    return chunks


# --- Apply chunking to all documents ---
# We store chunks as a list of dicts so we keep track of which
# document each chunk came from — critical for citations later

all_chunks = []  # list of dicts: {text, source, chunk_id}

for doc_name, doc_text in documents.items():
    chunks = recursive_chunk(doc_text, chunk_size=800, chunk_overlap=150)

    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "text": chunk,
            "source": doc_name,   # which PDF this came from
            "chunk_id": f"{doc_name}_chunk_{i}"  # unique ID
        })

# --- Sanity check ---
print(f" Total chunks created: {len(all_chunks)}")
print(f"\nBreakdown by document:")

# Count chunks per document
from collections import Counter
source_counts = Counter(c["source"] for c in all_chunks)
for source, count in source_counts.items():
    print(f"  {source}: {count} chunks")

# Preview one chunk so you can see what they look like
print(f"\n--- Sample chunk (chunk #50) ---")
print(all_chunks[50]["text"])
print(f"\nSource: {all_chunks[50]['source']}")
print(f"Length: {len(all_chunks[50]['text'])} characters")

 Total chunks created: 233

Breakdown by document:
  ETYS24 Publication.pdf: 131 chunks
  neso-winter-outlook-2025-26.pdf: 39 chunks
  future-energy-scenarios-2025-executive-summary_0.pdf: 41 chunks
  OFG2296_State of the Market Report.pdf: 22 chunks

--- Sample chunk (chunk #50) ---
different transfer capabilities.
longer term.
The boundary capability is limited to 1.1 GW due to a
thermal constraint on Beauly - Shin 132kV circuit

Boundary B1a – North West SSEN Transmission
The capability line (in red) is based on the recommendations from the Beyond 2030
optimal path which uses the 2023 FES and ETYS data as inputs. The 50%, 90%, Economy RT
2 and Security RT lines are based on the Clean Power 2030 pathways. T
o 4 2

Source: ETYS24 Publication.pdf
Length: 439 characters


In [15]:
# ============================================================
# CELL 4 — Step 1c: Embedding
# ============================================================

from sentence_transformers import SentenceTransformer
import numpy as np

# --- Load the embedding model ---
# all-MiniLM-L6-v2 is small (80MB), fast, and works well for
# domain-specific retrieval tasks. Downloads once, cached after.
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded")

# --- Extract just the text from each chunk ---
# The embedder needs a list of strings, not our dict objects
chunk_texts = [chunk["text"] for chunk in all_chunks]

# --- Generate embeddings ---
# show_progress_bar=True gives you a live progress bar in Colab
# batch_size=64 means it processes 64 chunks at a time (memory efficient)
print(f"\nEmbedding {len(chunk_texts)} chunks...")

embeddings = embedder.encode(
    chunk_texts,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True   # returns numpy array, which FAISS needs
)

# --- Sanity check ---
print(f"\n Embeddings created")
print(f"   Shape: {embeddings.shape}")
# Expected: (403, 384)
# 403 = number of chunks, 384 = dimensions per vector
print(f"   Each chunk is represented as {embeddings.shape[1]} numbers")
print(f"   Data type: {embeddings.dtype}")

# --- Quick similarity demo (optional but educational) ---
# Let's see if semantically similar chunks are actually close
from sklearn.metrics.pairwise import cosine_similarity

query_demo = "interconnector capacity winter"
query_vec = embedder.encode([query_demo], convert_to_numpy=True)

# Compute similarity between query and all chunks
sims = cosine_similarity(query_vec, embeddings)[0]
top_idx = sims.argsort()[-3:][::-1]  # top 3 most similar

print(f"\n--- Mini retrieval demo ---")
print(f"Query: '{query_demo}'\n")
for idx in top_idx:
    print(f"Score: {sims[idx]:.3f} | Source: {all_chunks[idx]['source']}")
    print(f"Text preview: {all_chunks[idx]['text'][:150]}...")
    print()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded

Embedding 233 chunks...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]


 Embeddings created
   Shape: (233, 384)
   Each chunk is represented as 384 numbers
   Data type: float32

--- Mini retrieval demo ---
Query: 'interconnector capacity winter'

Score: 0.614 | Source: neso-winter-outlook-2025-26.pdf
Text preview: operational surplus is sufficient
Nemo Link
(1 GW)
ElecLink
Belgium
Figure 9: Interconnectors operational this winter (1 GW)
(2 GW)
IFA2 France
(1 GW)...

Score: 0.567 | Source: neso-winter-outlook-2025-26.pdf
Text preview: export capacity on every interconnector with long-term auctions. Actual
flows will be driven by prices and prevailing conditions closer to real time.
...

Score: 0.559 | Source: ETYS24 Publication.pdf
Text preview: north to south. The arrows in the diagram illustrate power el
e flow directions and are approximately scaled relative to the winter peak flows. s
la &...



In [16]:
# ============================================================
# CELL 5 — Step 1d: FAISS Index + Excel Ingestion + Rebuild
# ============================================================
# Three sub-steps combined:
#   (i)   Build initial FAISS index from PDF chunks
#   (ii)  Ingest NESO Winter Outlook Excel workbook → text chunks
#   (iii) Merge Excel chunks into all_chunks, rebuild FAISS + save
#
# Why Excel ingestion here?
#   The PDFs miss numerical data from image-based tables (capacity MW,
#   demand forecasts, surplus figures). The Excel workbook contains the
#   exact figures behind every chart — adding it directly fills this gap.

import faiss
import numpy as np
import pickle
from openpyxl import load_workbook


# ── PART A: Build initial FAISS index from PDF chunks ────────────────────────

dimension = embeddings.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f" Initial FAISS index built")
print(f"   Vectors stored: {index.ntotal} | Dimensions: {dimension}")


# ── PART B: Excel ingestion ───────────────────────────────────────────────────

def excel_to_text_chunks(filepath):
    """
    Reads NESO Winter Outlook Excel workbook and converts key sheets
    into natural language text chunks with exact numerical figures.

    Why natural language?
      Both FAISS (semantic) and BM25 (keyword) operate on text.
      Converting rows like ('Nuclear', 6076, 4589) into
      'Nuclear: technical capacity 6,076 MW, de-rated 4,590 MW'
      makes the numbers retrievable by both systems.

    Returns: list of chunk dicts matching the existing all_chunks format.
    """
    wb           = load_workbook(filepath, read_only=True)
    excel_chunks = []
    source_name  = 'NESO_Winter_Outlook_Data_Workbook_2025-26.xlsx'
    chunk_counter = 0

    def safe_nums(row):
        """Extract all numeric values from a row, skipping None and strings."""
        return [v for v in row[1:] if isinstance(v, (int, float))]

    def add_chunk(text):
        nonlocal chunk_counter
        excel_chunks.append({
            'text'    : text,
            'source'  : source_name,
            'chunk_id': f'{source_name}_chunk_{chunk_counter}'
        })
        chunk_counter += 1

    # ── Figure 1: Capacity by fuel type ──────────────────────────────────────
    ws   = wb['Figure 1']
    rows = [r for r in ws.iter_rows(values_only=True)
            if any(v is not None for v in r)]

    lines = []
    for row in rows[2:]:  # skip title row and header row
        # Only process rows where col[1] and col[2] are actually numeric
        # This skips the column header row ('Type', 'Technical capability', ...)
        if row[0] and isinstance(row[1], (int, float)) and isinstance(row[2], (int, float)):
            lines.append(
                f"{row[0]}: technical capacity {float(row[1]):,.0f} MW, "
                f"de-rated capacity {float(row[2]):,.0f} MW"
            )
    for row in rows[2:]:
        if isinstance(row[3], (int, float)):
            lines.append(f"ACS peak demand: {float(row[3]):,.0f} MW"); break
    for row in rows[2:]:
        if isinstance(row[4], (int, float)):
            lines.append(f"De-rated capacity margin: {float(row[4]):,.2f} GW"); break

    add_chunk(
        "NESO Winter Outlook 2025/26 - Figure 1: "
        "De-rated Capacity Summary during system stress:\n" + "\n".join(lines)
    )

    # ── Figure 6: Peak demand forecast summary ────────────────────────────────
    ws   = wb['Figure 6']
    rows = [r for r in ws.iter_rows(values_only=True)
            if any(v is not None for v in r)]

    header_idx = next((i for i, r in enumerate(rows) if r[0] == 'Date'), None)
    if header_idx is not None:
        data_rows   = rows[header_idx + 1:]
        lower_row   = next((r for r in data_rows
                            if isinstance(r[0], str) and 'lower' in r[0].lower()), None)
        upper_row   = next((r for r in data_rows
                            if isinstance(r[0], str) and 'upper' in r[0].lower()), None)
        central_row = next((r for r in data_rows
                            if isinstance(r[0], str) and 'central' in r[0].lower()), None)

        lines = []
        if lower_row:
            vals = safe_nums(lower_row)
            if vals:
                lines.append(f"Minimum lower bound forecast (tightest cold day): {min(vals):,.0f} MW")
                lines.append(f"Maximum lower bound forecast: {max(vals):,.0f} MW")
        if upper_row:
            vals = safe_nums(upper_row)
            if vals:
                lines.append(f"Maximum upper bound forecast (peak demand): {max(vals):,.0f} MW")
        if central_row:
            vals = safe_nums(central_row)
            if vals:
                lines.append(f"Central forecast peak demand: {max(vals):,.0f} MW")
                lines.append(f"Central forecast minimum demand (holiday period): {min(vals):,.0f} MW")

        add_chunk(
            "NESO Winter Outlook 2025/26 - Figure 6: "
            "Peak National Demand Forecast (Oct 2025 – Mar 2026):\n" + "\n".join(lines)
        )

    # ── Generation and Supply: central forecast summary ───────────────────────
    ws   = wb['Generation and supply']
    rows = [r for r in ws.iter_rows(values_only=True)
            if any(v is not None for v in r)]

    header_idx = next((i for i, r in enumerate(rows) if r[0] == 'Date'), None)
    if header_idx is not None:
        data_rows   = rows[header_idx + 1:]
        demand_row  = next((r for r in data_rows
                            if isinstance(r[0], str)
                            and 'national demand' in r[0].lower()
                            and 'central' in r[0].lower()), None)
        reserve_row = next((r for r in data_rows
                            if isinstance(r[0], str) and 'reserve' in r[0].lower()), None)
        gen_row     = next((r for r in data_rows
                            if isinstance(r[0], str)
                            and 'transmission generation' in r[0].lower()), None)

        lines = []
        if demand_row:
            vals = safe_nums(demand_row)
            if vals:
                lines.append(f"Central forecast peak national demand (inc. station load): {max(vals):,.0f} MW")
                lines.append(f"Central forecast minimum demand: {min(vals):,.0f} MW")
        if reserve_row:
            vals = safe_nums(reserve_row)
            if vals:
                lines.append(f"Assumed reserve requirement: {vals[0]:,.0f} MW (constant throughout winter)")
        if gen_row:
            vals = safe_nums(gen_row)
            if vals:
                lines.append(f"Maximum assumed transmission generation (no interconnector imports): {max(vals):,.0f} MW")
                lines.append(f"Minimum assumed transmission generation: {min(vals):,.0f} MW")

        add_chunk(
            "NESO Winter Outlook 2025/26 - Generation and Supply Central Forecast Summary:\n"
            + "\n".join(lines)
        )

    # ── Figure 2: Operational surplus summary ─────────────────────────────────
    ws   = wb['Figure 2']
    rows = [r for r in ws.iter_rows(values_only=True)
            if any(v is not None for v in r)]

    header_idx = next((i for i, r in enumerate(rows) if r[0] == 'Date'), None)
    if header_idx is not None:
        data_rows   = rows[header_idx + 1:]
        lower_row   = next((r for r in data_rows
                            if isinstance(r[0], str) and 'lower' in r[0].lower()), None)
        upper_row   = next((r for r in data_rows
                            if isinstance(r[0], str) and 'upper' in r[0].lower()), None)
        central_row = next((r for r in data_rows
                            if isinstance(r[0], str) and 'central' in r[0].lower()), None)

        lines = []
        if lower_row:
            vals = safe_nums(lower_row)
            if vals:
                lines.append(f"Minimum operational surplus (tightest supply margin): {min(vals):,.0f} MW")
        if upper_row:
            vals = safe_nums(upper_row)
            if vals:
                lines.append(f"Maximum operational surplus: {max(vals):,.0f} MW")
        if central_row:
            vals = safe_nums(central_row)
            if vals:
                lines.append(f"Central forecast operational surplus peak: {max(vals):,.0f} MW")
                lines.append(f"Central forecast operational surplus minimum: {min(vals):,.0f} MW")

        add_chunk(
            "NESO Winter Outlook 2025/26 - Figure 2: "
            "Operational Surplus Forecast (Base Case, Oct 2025 – Mar 2026):\n"
            + "\n".join(lines)
        )

    print(f" Excel ingestion complete: {len(excel_chunks)} chunks created")
    for c in excel_chunks:
        print(f"  {c['chunk_id']}: {len(c['text'])} chars")
        print(f"  Preview: {c['text'][:120]}\n")

    return excel_chunks


# --- Run Excel ingestion ---
# Add any Excel workbook paths to this list — all will be ingested automatically
EXCEL_FILES = [
    '/content/sample_data/data/2025-26 NESO Winter Outlook Data Workbook.xlsx',
    # '/content/sample_data/data/another_workbook.xlsx',  # add more here
]

excel_chunks = []
for excel_path in EXCEL_FILES:
    excel_chunks.extend(excel_to_text_chunks(excel_path))

print(f"\n Total Excel chunks across all workbooks: {len(excel_chunks)}")

# ── PART C: Merge and rebuild index ──────────────────────────────────────────
# Add Excel chunks to the existing all_chunks list
all_chunks.extend(excel_chunks)

# Re-embed the new Excel chunks only (efficient — don't re-embed PDFs)
excel_texts       = [c['text'] for c in excel_chunks]
excel_embeddings  = embedder.encode(
    excel_texts,
    show_progress_bar=False,
    convert_to_numpy=True
)

# Stack PDF embeddings + Excel embeddings into one array
all_embeddings = np.vstack([embeddings, excel_embeddings])

# Rebuild FAISS index with the full corpus
index = faiss.IndexFlatL2(dimension)
index.add(all_embeddings.astype(np.float32))  # ensure float32 for FAISS

print(f"\n Index rebuilt with full corpus")
print(f"   Total chunks: {len(all_chunks)} (PDF: {len(all_chunks) - len(excel_chunks)}, Excel: {len(excel_chunks)})")
print(f"   FAISS vectors: {index.ntotal}")

# --- Save everything to disk ---
faiss.write_index(index, '/content/energy_rag.index')

with open('/content/chunk_metadata.pkl', 'wb') as f:
    pickle.dump(all_chunks, f)

print("\n Index saved to: /content/energy_rag.index")
print(" Metadata saved to: /content/chunk_metadata.pkl")
print("\nStep 1 complete — full corpus indexed and ready for retrieval.")


# --- Quick retrieval test on an Excel-sourced query ---
# This should now return the Excel chunk with exact MW figures
test_query = "What is the peak national demand forecast for winter 2025/26?"
test_vec   = embedder.encode([test_query], convert_to_numpy=True)
D, I       = index.search(test_vec, 3)

print(f"\n--- Quick test: Excel data retrieval ---")
print(f"Query: '{test_query}'\n")
for rank, (dist, idx) in enumerate(zip(D[0], I[0])):
    print(f"Rank {rank+1} | Distance: {dist:.3f} | Source: {all_chunks[idx]['source']}")
    print(f"Preview: {all_chunks[idx]['text'][:200]}")
    print()


 Initial FAISS index built
   Vectors stored: 233 | Dimensions: 384
 Excel ingestion complete: 4 chunks created
  NESO_Winter_Outlook_Data_Workbook_2025-26.xlsx_chunk_0: 562 chars
  Preview: NESO Winter Outlook 2025/26 - Figure 1: De-rated Capacity Summary during system stress:
Nuclear: technical capacity 6,07

  NESO_Winter_Outlook_Data_Workbook_2025-26.xlsx_chunk_1: 346 chars
  Preview: NESO Winter Outlook 2025/26 - Figure 6: Peak National Demand Forecast (Oct 2025 – Mar 2026):
Minimum lower bound forecas

  NESO_Winter_Outlook_Data_Workbook_2025-26.xlsx_chunk_2: 386 chars
  Preview: NESO Winter Outlook 2025/26 - Generation and Supply Central Forecast Summary:
Central forecast peak national demand (inc

  NESO_Winter_Outlook_Data_Workbook_2025-26.xlsx_chunk_3: 312 chars
  Preview: NESO Winter Outlook 2025/26 - Figure 2: Operational Surplus Forecast (Base Case, Oct 2025 – Mar 2026):
Minimum operation


 Total Excel chunks across all workbooks: 4

 Index rebuilt with full corpus
   Tot

In [17]:
# ============================================================
# CELL 6 — Step 2: Hybrid Retrieval (BM25 + FAISS + RRF)
# ============================================================

from rank_bm25 import BM25Okapi  # BM25 implementation
import numpy as np

# --- Build BM25 index ---
# BM25 works on tokens (individual words), not vectors
# So we tokenise each chunk by splitting on whitespace
# Simple but effective for technical documents

print("Building BM25 index...")

# Tokenise: lowercase each chunk, split into words
tokenised_chunks = [
    chunk["text"].lower().split()
    for chunk in all_chunks
]

# Build the BM25 index from tokenised chunks
bm25 = BM25Okapi(tokenised_chunks)

print(f"BM25 index built over {len(tokenised_chunks)} chunks")


# --- Define individual retrieval functions ---

def faiss_search(query, k=20):
    """
    Retrieve top-k chunks using FAISS vector search.
    Returns list of (chunk_index, distance) tuples.
    Lower distance = better match.
    """
    query_vec = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vec, k)

    # Return as list of (index, distance) pairs
    return list(zip(indices[0], distances[0]))


def bm25_search(query, k=20):
    """
    Retrieve top-k chunks using BM25 keyword search.
    Returns list of (chunk_index, score) tuples.
    Higher score = better match.
    """
    # Tokenise the query the same way we tokenised chunks
    tokenised_query = query.lower().split()

    # Get BM25 scores for all chunks
    scores = bm25.get_scores(tokenised_query)

    # Get indices of top-k scores
    top_indices = np.argsort(scores)[::-1][:k]

    return [(idx, scores[idx]) for idx in top_indices]


def reciprocal_rank_fusion(faiss_results, bm25_results, k=60):
    """
    Combines FAISS and BM25 results using Reciprocal Rank Fusion.

    RRF formula: score(chunk) = sum of 1/(rank + k) across all systems
    k=60 is standard — dampens the impact of very high ranks

    Returns: list of chunk indices sorted by combined RRF score
    """
    rrf_scores = {}  # {chunk_index: combined_rrf_score}

    # Add RRF scores from FAISS results
    # enumerate gives us (rank_position, (chunk_idx, distance))
    for rank, (chunk_idx, _) in enumerate(faiss_results):
        if chunk_idx not in rrf_scores:
            rrf_scores[chunk_idx] = 0
        rrf_scores[chunk_idx] += 1 / (rank + 1 + k)

    # Add RRF scores from BM25 results
    for rank, (chunk_idx, _) in enumerate(bm25_results):
        if chunk_idx not in rrf_scores:
            rrf_scores[chunk_idx] = 0
        rrf_scores[chunk_idx] += 1 / (rank + 1 + k)

    # Sort by combined score — higher is better
    sorted_results = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return sorted_results  # list of (chunk_idx, rrf_score)


def hybrid_search(query, top_k=5):
    """
    Full hybrid retrieval pipeline.
    1. Get top-20 from FAISS
    2. Get top-20 from BM25
    3. Fuse with RRF
    4. Return top_k results with chunk text
    """
    # Step 1 & 2: retrieve from both systems
    faiss_results = faiss_search(query, k=20)
    bm25_results  = bm25_search(query, k=20)

    # Step 3: fuse rankings
    fused = reciprocal_rank_fusion(faiss_results, bm25_results)

    # Step 4: return top_k with full chunk data
    top_results = []
    for chunk_idx, rrf_score in fused[:top_k]:
        top_results.append({
            "chunk"    : all_chunks[chunk_idx],
            "rrf_score": rrf_score,
            "chunk_idx": chunk_idx
        })

    return top_results


# ============================================================
# TEST 1 — Semantic query (vector search should shine)
# ============================================================
print("\n" + "="*60)
print("TEST 1 — Semantic query")
print("="*60)
query1 = "What is the outlook for electricity supply this winter?"
results1 = hybrid_search(query1, top_k=5)

print(f"Query: '{query1}'\n")
for i, r in enumerate(results1):
    print(f"Rank {i+1} | RRF: {r['rrf_score']:.4f} | Source: {r['chunk']['source']}")
    print(f"Preview: {r['chunk']['text'][:200]}")
    print()

# ============================================================
# TEST 2 — Exact term query (BM25 should shine)
# ============================================================
print("="*60)
print("TEST 2 — Exact term query")
print("="*60)
query2 = "Greenlink interconnector 0.5 GW capacity"
results2 = hybrid_search(query2, top_k=5)

print(f"Query: '{query2}'\n")
for i, r in enumerate(results2):
    print(f"Rank {i+1} | RRF: {r['rrf_score']:.4f} | Source: {r['chunk']['source']}")
    print(f"Preview: {r['chunk']['text'][:200]}")
    print()

Building BM25 index...
BM25 index built over 237 chunks

TEST 1 — Semantic query
Query: 'What is the outlook for electricity supply this winter?'

Rank 1 | RRF: 0.0318 | Source: neso-winter-outlook-2025-26.pdf
Preview: ational Surplus Analysis 29
Demand and Supply 14

Peak demand and credible range 15
Glossary 30
Conventional generator availability 16
Get in Touch 34

Welcome to the Winter Outlook 2025/26 report, As

Rank 2 | RRF: 0.0315 | Source: neso-winter-outlook-2025-26.pdf
Preview: nce bound Central forecast
Figure 2: Range of outcomes for the daily operational surplus in our base case under different
supply and demand conditions

winter. The red shaded region shows the credible

Rank 3 | RRF: 0.0310 | Source: neso-winter-outlook-2025-26.pdf
Preview: these additional insights support industry stakeholders to
days that may require the use
plan and prepare more effectively.
of routine system notices.

4/Winter Outlook 2025–26/Key messages: Winter Ou

Rank 4 | RRF: 0.0306 | Source: 

In [18]:
# ============================================================
# CELL 7 — Step 3: Cross-Encoder Reranker
# ============================================================

from sentence_transformers import CrossEncoder

# --- Load the cross-encoder model ---
# ms-marco-MiniLM-L-6-v2 is trained specifically on
# (query, passage) relevance scoring — perfect for RAG reranking
# It outputs a single relevance score per (query, chunk) pair
print("Loading cross-encoder reranker...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("Reranker loaded")


def rerank(query, candidate_chunks, top_k=5):
    """
    Takes a query and a list of candidate chunks from hybrid search.
    Cross-encoder scores each (query, chunk) pair together.
    Returns top_k chunks reordered by true relevance.

    candidate_chunks: list of dicts from hybrid_search()
    """
    # Build (query, chunk_text) pairs for the cross-encoder
    # The model reads each pair jointly to score relevance
    pairs = [
        (query, candidate["chunk"]["text"])
        for candidate in candidate_chunks
    ]

    # Score all pairs — returns a list of relevance scores
    # Higher score = more relevant
    scores = reranker.predict(pairs)

    # Attach scores back to each candidate
    for i, candidate in enumerate(candidate_chunks):
        candidate["rerank_score"] = float(scores[i])

    # Sort by rerank score — highest first
    reranked = sorted(
        candidate_chunks,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return reranked[:top_k]


def full_pipeline(query, retrieve_k=20, final_k=5):
    """
    Complete enhanced RAG retrieval pipeline:
    1. Hybrid search (FAISS + BM25 + RRF) → top retrieve_k candidates
    2. Cross-encoder reranker → top final_k results

    retrieve_k: how many candidates to pass to reranker
    final_k: how many final results to return
    """
    # Stage 1: hybrid retrieval — cast wide net
    candidates = hybrid_search(query, top_k=retrieve_k)

    # Stage 2: rerank — apply precision
    reranked = rerank(query, candidates, top_k=final_k)

    return reranked


# ============================================================
# TEST 1 — Semantic query (same as Cell 6 Test 1)
# ============================================================
print("\n" + "="*60)
print("TEST 1 — Semantic query (with reranker)")
print("="*60)
query1 = "What is the outlook for electricity supply this winter?"
results1 = full_pipeline(query1)

print(f"Query: '{query1}'\n")
for i, r in enumerate(results1):
    print(f"Rank {i+1} | Rerank: {r['rerank_score']:.3f} | RRF: {r['rrf_score']:.4f} | Source: {r['chunk']['source']}")
    print(f"Preview: {r['chunk']['text'][:250]}")
    print()

# ============================================================
# TEST 2 — Exact term query (same as Cell 6 Test 2)
# ============================================================
print("="*60)
print("TEST 2 — Exact term query (with reranker)")
print("="*60)
query2 = "Greenlink interconnector 0.5 GW capacity"
results2 = full_pipeline(query2)

print(f"Query: '{query2}'\n")
for i, r in enumerate(results2):
    print(f"Rank {i+1} | Rerank: {r['rerank_score']:.3f} | RRF: {r['rrf_score']:.4f} | Source: {r['chunk']['source']}")
    print(f"Preview: {r['chunk']['text'][:250]}")
    print()

# ============================================================
# TEST 3 — Cross-document query (should pull from multiple docs)
# ============================================================
print("="*60)
print("TEST 3 — Cross-document query")
print("="*60)
query3 = "What are the key risks to UK energy security in the transition to net zero?"
results3 = full_pipeline(query3)

print(f"Query: '{query3}'\n")
for i, r in enumerate(results3):
    print(f"Rank {i+1} | Rerank: {r['rerank_score']:.3f} | RRF: {r['rrf_score']:.4f} | Source: {r['chunk']['source']}")
    print(f"Preview: {r['chunk']['text'][:250]}")
    print()

Loading cross-encoder reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Reranker loaded

TEST 1 — Semantic query (with reranker)
Query: 'What is the outlook for electricity supply this winter?'

Rank 1 | Rerank: 5.490 | RRF: 0.0297 | Source: neso-winter-outlook-2025-26.pdf
Preview: Winter Outlook 2025/26
Helping to inform the electricity industry
and prepare for the winter ahead
October 2025
National Energy
System Operator

Key messages: Winter Outlook Appendix A: List of Figures and Tables 25
Operational Surplus 10 Appendix B:

Rank 2 | Rerank: 4.934 | RRF: 0.0318 | Source: neso-winter-outlook-2025-26.pdf
Preview: ational Surplus Analysis 29
Demand and Supply 14

Peak demand and credible range 15
Glossary 30
Conventional generator availability 16
Get in Touch 34

Welcome to the Winter Outlook 2025/26 report, As a prudent system operator, our planning and We wa

Rank 3 | Rerank: 3.860 | RRF: 0.0310 | Source: neso-winter-outlook-2025-26.pdf
Preview: these additional insights support industry stakeholders to
days that may require the use
plan and prepare mor

In [19]:
!pip install anthropic -q

In [20]:
# ============================================================
# CELL 8 — Fix 4: LLM Query Rewriting + HyDE
# ============================================================
# Two query enhancement strategies implemented:
#
#   (i)  Query Rewriting — Claude expands the user query with
#        domain-specific terminology to improve BM25 + FAISS recall
#
#  (ii)  HyDE (Hypothetical Document Embedding) — Claude generates
#        a hypothetical answer, which is then embedded and used as
#        the FAISS search vector instead of the original query.
#        Rationale: a hypothetical answer resembles a document chunk
#        more than a question does, reducing the query-document gap.
#
# Both strategies run before retrieval — they are query-side
# improvements that require no changes to the index or corpus.

import anthropic
import os

# --- Initialise Anthropic client ---
# Set your API key: Runtime → Secrets → Add 'ANTHROPIC_API_KEY'
# or paste directly (not recommended for shared notebooks)
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic()


def rewrite_query(query):
    """
    Strategy 1: Query Rewriting
    Sends the user query to Claude and asks it to expand it with
    domain-specific UK energy sector terminology.

    Why this helps:
      Users write conversational queries; documents contain technical
      jargon. Rewriting bridges the vocabulary gap for both BM25
      (keyword matching) and FAISS (semantic similarity).
    """
    prompt = f"""You are a UK energy sector expert assistant helping to improve
document retrieval from NESO and Ofgem reports.

Rewrite the following query to improve retrieval from technical energy
documents. Expand it with relevant UK energy sector terminology, acronyms,
and technical concepts that would appear in NESO/Ofgem reports.

Return ONLY the rewritten query in 20-30 words maximum — no explanation, no preamble.

Original query: {query}

Rewritten query:"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=150,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()


def hyde_query(query):
    """
    Strategy 2: HyDE — Hypothetical Document Embedding
    Asks Claude to generate a hypothetical passage that would answer
    the query, as if it were extracted from a NESO/Ofgem report.

    The hypothetical passage is then embedded (not the original query)
    and used for FAISS vector search.

    Why this helps:
      The embedding space was trained on document-like text.
      A hypothetical answer vector sits much closer to real document
      chunk vectors than a question vector does — improving recall
      for semantic search significantly.
    """
    prompt = f"""You are a UK energy sector analyst. Write a short passage
(3-5 sentences) as if it were extracted from a NESO or Ofgem technical
report that directly answers the following question.

Use realistic technical language, figures, and terminology that would
appear in such a report. Do not mention that this is hypothetical.

Question: {query}

Passage:"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()


def enhanced_pipeline(query, retrieve_k=20, final_k=5, strategy="rewrite"):
    """
    Full enhanced RAG retrieval pipeline with query enhancement.

    Strategies:
      'rewrite' — expand query with domain terminology before hybrid search
      'hyde'    — generate hypothetical answer, embed it for FAISS search
      'both'    — apply rewriting for BM25, HyDE embedding for FAISS

    Pipeline:
      1. Enhance query (rewrite or HyDE)
      2. Hybrid search (FAISS + BM25 + RRF) → top retrieve_k
      3. Cross-encoder reranker → top final_k
    """
    enhanced_query = query  # fallback

    if strategy == "rewrite":
        enhanced_query = rewrite_query(query)
        print(f"  [Rewritten]: {enhanced_query}\n")
        candidates = hybrid_search(enhanced_query, top_k=retrieve_k)

    elif strategy == "hyde":
        hyde_passage = hyde_query(query)
        print(f"  [HyDE passage]: {hyde_passage[:200]}...\n")
        # Embed the hypothetical passage for FAISS
        hyde_vec      = embedder.encode([hyde_passage], convert_to_numpy=True)
        D, I          = index.search(hyde_vec, retrieve_k)
        faiss_results = list(zip(I[0], D[0]))
        # Still use original query for BM25 (keyword matching)
        bm25_results  = bm25_search(query, k=retrieve_k)
        fused         = reciprocal_rank_fusion(faiss_results, bm25_results)
        candidates    = [
            {"chunk": all_chunks[idx], "rrf_score": score, "chunk_idx": idx}
            for idx, score in fused[:retrieve_k]
        ]

    elif strategy == "both":
        # Rewrite for BM25, HyDE for FAISS
        enhanced_query = rewrite_query(query)
        hyde_passage   = hyde_query(query)
        print(f"  [Rewritten]: {enhanced_query}")
        print(f"  [HyDE]:      {hyde_passage[:150]}...\n")
        hyde_vec      = embedder.encode([hyde_passage], convert_to_numpy=True)
        D, I          = index.search(hyde_vec, retrieve_k)
        faiss_results = list(zip(I[0], D[0]))
        bm25_results  = bm25_search(enhanced_query, k=retrieve_k)
        fused         = reciprocal_rank_fusion(faiss_results, bm25_results)
        candidates    = [
            {"chunk": all_chunks[idx], "rrf_score": score, "chunk_idx": idx}
            for idx, score in fused[:retrieve_k]
        ]

    # Rerank with cross-encoder — always uses original query
    # (cross-encoder compares original question vs chunk, not rewritten)
    reranked = rerank(query, candidates, top_k=final_k)
    return reranked


# ============================================================
# TEST 1 — Strategy comparison on semantic query
# ============================================================
print("=" * 60)
print("TEST 1 — Semantic query: rewrite vs HyDE vs both")
print("=" * 60)
query1 = "What is the outlook for electricity supply this winter?"

for strategy in ["rewrite", "hyde", "both"]:
    print(f"\n--- Strategy: {strategy.upper()} ---")
    results = enhanced_pipeline(query1, strategy=strategy)
    for i, r in enumerate(results[:3]):  # show top 3 for brevity
        print(f"  Rank {i+1} | Rerank: {r['rerank_score']:.3f} | "
              f"Source: {r['chunk']['source']}")
        print(f"  Preview: {r['chunk']['text'][:180]}")
        print()

# ============================================================
# TEST 2 — Cross-document synthesis query
# ============================================================
print("=" * 60)
print("TEST 2 — Cross-document query with 'both' strategy")
print("=" * 60)
query2 = "What are the key risks to UK energy security in the transition to net zero?"
print(f"Query: '{query2}'\n")

results2 = enhanced_pipeline(query2, strategy="both")
for i, r in enumerate(results2):
    print(f"Rank {i+1} | Rerank: {r['rerank_score']:.3f} | "
          f"Source: {r['chunk']['source']}")
    print(f"Preview: {r['chunk']['text'][:200]}")
    print()

# ============================================================
# TEST 3 — Excel data query
# ============================================================
print("=" * 60)
print("TEST 3 — Numerical data query with 'both' strategy")
print("=" * 60)
query3 = "What is the peak national demand forecast for winter 2025/26?"
print(f"Query: '{query3}'\n")

results3 = enhanced_pipeline(query3, strategy="both")
for i, r in enumerate(results3):
    print(f"Rank {i+1} | Rerank: {r['rerank_score']:.3f} | "
          f"Source: {r['chunk']['source']}")
    print(f"Preview: {r['chunk']['text'][:200]}")
    print()

TEST 1 — Semantic query: rewrite vs HyDE vs both

--- Strategy: REWRITE ---
  [Rewritten]: Winter Outlook Report electricity supply adequacy margin capacity assessment demand forecast generation availability ESO NESO security of supply peak demand de-rated capacity reserve margins MCR LOLP.

  Rank 1 | Rerank: 5.490 | Source: neso-winter-outlook-2025-26.pdf
  Preview: Winter Outlook 2025/26
Helping to inform the electricity industry
and prepare for the winter ahead
October 2025
National Energy
System Operator

Key messages: Winter Outlook Append

  Rank 2 | Rerank: 4.934 | Source: neso-winter-outlook-2025-26.pdf
  Preview: ational Surplus Analysis 29
Demand and Supply 14

Peak demand and credible range 15
Glossary 30
Conventional generator availability 16
Get in Touch 34

Welcome to the Winter Outloo

  Rank 3 | Rerank: 3.860 | Source: neso-winter-outlook-2025-26.pdf
  Preview: these additional insights support industry stakeholders to
days that may require the use
plan and prepare mor

In [21]:
# ============================================================
# CELL 9 — Step 4: Generation with Anthropic API
# ============================================================
# Two generation pipelines implemented side by side:
#
#   Baseline  — simple FAISS retrieval (top 3) + vanilla prompt
#               Represents a naive RAG implementation
#
#   Enhanced  — HyDE + hybrid retrieval + reranking (top 5)
#               + grounded prompt with citation instructions
#               Represents our full advanced RAG system
#
# Having both in one cell makes baseline vs enhanced comparison
# easy to run and document for the assignment evaluation.

import anthropic
from google.colab import userdata

# --- Initialise client ---
client = anthropic.Anthropic(
    api_key=userdata.get('ANTHROPIC_API_KEY')
)

GENERATION_MODEL = "claude-haiku-4-5"  # fast + cheap for evaluation runs
                                        # swap to claude-sonnet-4-6 for
                                        # higher quality final demo


# ── BASELINE PIPELINE ────────────────────────────────────────────────────────

def baseline_retrieve(query, k=3):
    """
    Simple FAISS-only retrieval — no hybrid, no reranking.
    Represents the naive RAG baseline.
    """
    query_vec  = embedder.encode([query], convert_to_numpy=True)
    D, I       = index.search(query_vec, k)
    return [all_chunks[idx] for idx in I[0]]


def baseline_generate(query):
    """
    Baseline generation:
      - Simple FAISS retrieval (top 3 chunks)
      - Minimal prompt with no grounding instructions
      - No citation requirements

    This intentionally represents a weak RAG implementation
    to contrast against the enhanced pipeline.
    """
    chunks = baseline_retrieve(query, k=3)

    # Build context string — just concatenate chunks
    context = "\n\n".join([
        f"[Source: {c['source']}]\n{c['text']}"
        for c in chunks
    ])

    # Vanilla prompt — minimal instructions
    prompt = f"""Use the following documents to answer the question.

{context}

Question: {query}
Answer:"""

    response = client.messages.create(
        model=GENERATION_MODEL,
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}]
    )

    return {
        "answer"  : response.content[0].text.strip(),
        "chunks"  : chunks,
        "pipeline": "baseline"
    }


# ── ENHANCED PIPELINE ─────────────────────────────────────────────────────────

def enhanced_generate(query, strategy="both"):
    """
    Enhanced generation:
      - HyDE + query rewriting + hybrid retrieval + cross-encoder reranking
      - Grounded system prompt with explicit citation instructions
      - Instructed to acknowledge gaps rather than hallucinate

    This is the full advanced RAG pipeline.
    """
    # Stage 1: enhanced retrieval
    reranked_chunks = enhanced_pipeline(
        query,
        retrieve_k=20,
        final_k=5,
        strategy=strategy
    )
    chunks = [r["chunk"] for r in reranked_chunks]

    # Stage 2: build numbered context with source labels
    # Numbered so Claude can cite by number in its answer
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        context_parts.append(
            f"[{i}] Source: {chunk['source']}\n{chunk['text']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    # Stage 3: grounded system prompt
    system_prompt = """You are a UK energy sector analyst assistant with expertise
in NESO and Ofgem reports, grid operations, and energy market analysis.

Your answers must follow these rules strictly:
1. Answer using ONLY information from the provided context chunks
2. Cite your sources using the chunk numbers e.g. [1], [2], [3]
3. If the context does not contain enough information to answer fully,
   explicitly state what is missing rather than inferring or guessing
4. For numerical figures (MW, GW, percentages), quote them exactly
   as they appear in the context — do not round or estimate
5. Structure your answer clearly with the most important finding first
6. Keep your answer concise — 150 words maximum"""

    # Stage 4: user prompt with context + question
    user_prompt = f"""Context chunks from NESO and Ofgem documents:

{context}

Question: {query}

Answer (cite sources using [1], [2] etc.):"""

    response = client.messages.create(
        model=GENERATION_MODEL,
        max_tokens=400,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}]
    )

    return {
        "answer"  : response.content[0].text.strip(),
        "chunks"  : chunks,
        "pipeline": "enhanced"
    }


# ── COMPARISON FUNCTION ───────────────────────────────────────────────────────

def compare_pipelines(query):
    """
    Runs both pipelines on the same query and prints results
    side by side for easy comparison.
    """
    print("=" * 70)
    print(f"QUERY: {query}")
    print("=" * 70)

    # Run baseline
    print("\n BASELINE PIPELINE")
    print("-" * 40)
    baseline = baseline_generate(query)
    print(baseline["answer"])
    print(f"\nSources used:")
    for c in baseline["chunks"]:
        print(f"  - {c['source']}")

    # Run enhanced
    print("\n ENHANCED PIPELINE")
    print("-" * 40)
    enhanced = enhanced_generate(query)
    print(enhanced["answer"])
    print(f"\nSources used:")
    for c in enhanced["chunks"]:
        print(f"  - {c['source']}")

    print("\n" + "=" * 70)
    return baseline, enhanced


# ── RUN COMPARISONS ───────────────────────────────────────────────────────────

# Query 1 — Numerical/factual (where enhanced should shine)
b1, e1 = compare_pipelines(
    "What is the peak national demand forecast for winter 2025/26?"
)

# Query 2 — Semantic (operational outlook)
b2, e2 = compare_pipelines(
    "What tools does NESO use to balance the electricity system?"
)

# Query 3 — Cross-document (net zero + grid)
b3, e3 = compare_pipelines(
    "What are the future energy scenarios for GB electricity demand?"
)

QUERY: What is the peak national demand forecast for winter 2025/26?

 BASELINE PIPELINE
----------------------------------------
# Peak National Demand Forecast for Winter 2025/26

Based on the documents provided, there are two peak national demand figures cited:

1. **Central forecast peak demand: 42,613 MW** (from Figure 6 data)

2. **Central forecast peak national demand (including station load): 43,881 MW** (from the Generation and Supply Central Forecast Summary)

The difference between these figures is due to the inclusion of station load in the second measurement. 

For the **maximum upper bound forecast (peak demand): 46,612 MW** represents the highest potential demand scenario under extreme conditions.

The most commonly referenced figure in the winter outlook is the **central forecast peak demand of approximately 42.6-43.9 GW**, with the possibility of demand exceeding 45 GW during periods of cold and calm weekday weather, and approximately a 5% chance that National Demand c

In [11]:
# # ============================================================
# # CELL 10 — Step 5: Evaluation Framework
# # ============================================================
# # Interactive evaluation system — add your own queries and
# # expected answers, then run both pipelines and score results.
# #
# # Evaluates:
# #   Retrieval  — MRR (Mean Reciprocal Rank)
# #   Generation — Faithfulness, Correctness, Completeness, Grounding
# #                scored 1-3 per dimension using Claude as judge
# #
# # Design:
# #   - Add queries to EVAL_QUERIES list below
# #   - Each query has: question, expected_answer, relevant_keywords
# #   - Framework runs baseline + enhanced on every query
# #   - Claude scores each answer automatically
# #   - Summary table printed at end

# import pandas as pd
# import json

# # ============================================================
# # EVALUATION QUERY SET
# # Add your own queries here — diverse types as required by brief
# # ============================================================
# # Query types covered (assignment requires diversity):
# #   - Simple facts        : single specific figure
# #   - Deep context        : requires synthesising multiple chunks
# #   - Ambiguous           : could mean multiple things
# #   - Edge cases          : topic at boundary of corpus coverage

# EVAL_QUERIES = [
#     # ── Simple factual queries ────────────────────────────────
#     {
#         "id"              : "Q01",
#         "type"            : "simple_fact",
#         "question"        : "What is the peak national demand forecast for winter 2025/26?",
#         "expected_answer" : "Central forecast peak demand is 42,613 MW with upper bound of 46,612 MW",
#         "keywords"        : ["42,613", "46,612", "MW", "peak", "demand"]
#     },
#     {
#         "id"              : "Q02",
#         "type"            : "simple_fact",
#         "question"        : "What is the capacity of the Greenlink interconnector?",
#         "expected_answer" : "0.5 GW interconnector between Ireland and Great Britain commissioned in early 2025",
#         "keywords"        : ["0.5", "GW", "Greenlink", "Ireland", "2025"]
#     },
#     {
#         "id"              : "Q03",
#         "type"            : "simple_fact",
#         "question"        : "What is the nuclear technical capacity in the de-rated capacity summary?",
#         "expected_answer" : "Nuclear technical capacity is 6,076 MW",
#         "keywords"        : ["6,076", "nuclear", "MW", "capacity"]
#     },

#     # ── Deep context queries ──────────────────────────────────
#     {
#         "id"              : "Q04",
#         "type"            : "deep_context",
#         "question"        : "What tools does NESO use to balance the electricity system during winter?",
#         "expected_answer" : "Balancing Mechanism, Demand Flexibility Service, reserve requirements, system notices",
#         "keywords"        : ["Balancing Mechanism", "Demand Flexibility", "reserve", "notices"]
#     },
#     {
#         "id"              : "Q05",
#         "type"            : "deep_context",
#         "question"        : "What are the future energy scenarios for GB electricity demand by 2050?",
#         "expected_answer" : "Four scenarios: System Transformation 705 TWh, Early Evolution 785 TWh, Headroom for Equity 797 TWh, Falling Behind 559 TWh",
#         "keywords"        : ["705", "785", "797", "559", "TWh", "2050"]
#     },
#     {
#         "id"              : "Q06",
#         "type"            : "deep_context",
#         "question"        : "What is the operational surplus forecast for winter 2025/26 and what drives year-on-year changes?",
#         "expected_answer" : "Operational surplus increased year-on-year driven by greater battery storage capacity and increased interconnector capacity including Greenlink",
#         "keywords"        : ["surplus", "battery", "storage", "interconnector", "margin"]
#     },

#     # ── Ambiguous queries ─────────────────────────────────────
#     {
#         "id"              : "Q07",
#         "type"            : "ambiguous",
#         "question"        : "How secure is UK energy this winter?",
#         "expected_answer" : "Overall assessment remains broadly similar to previous year with adequate margins; Greenlink adds 0.3 GW increase in de-rated margin",
#         "keywords"        : ["secure", "adequate", "margin", "winter", "supply"]
#     },
#     {
#         "id"              : "Q08",
#         "type"            : "ambiguous",
#         "question"        : "What is the energy market outlook?",
#         "expected_answer" : "Ofgem state of market report covers retail market, wholesale prices, network charges",
#         "keywords"        : ["market", "Ofgem", "retail", "wholesale", "outlook"]
#     },

#     # ── Edge case queries ─────────────────────────────────────
#     {
#         "id"              : "Q09",
#         "type"            : "edge_case",
#         "question"        : "What happens if there is a gas supply disruption during winter 2025/26?",
#         "expected_answer" : "Context may not fully cover this — tests corpus boundary",
#         "keywords"        : ["gas", "disruption", "supply", "contingency", "risk"]
#     },
#     {
#         "id"              : "Q10",
#         "type"            : "edge_case",
#         "question"        : "What is the carbon intensity of the GB electricity grid in 2025?",
#         "expected_answer" : "Context may not contain this — tests hallucination boundary",
#         "keywords"        : ["carbon", "intensity", "gCO2", "emissions", "grid"]
#     },
# ]


# # ============================================================
# # RETRIEVAL EVALUATION — MRR
# # ============================================================

# def compute_mrr(query_dict, pipeline="enhanced"):
#     """
#     Mean Reciprocal Rank — measures how highly the first
#     relevant chunk is ranked.

#     MRR = 1/rank of first relevant chunk
#       Rank 1 = 1.0  (perfect)
#       Rank 2 = 0.5
#       Rank 3 = 0.33
#       Not found = 0.0

#     Relevance determined by keyword matching against
#     expected_answer keywords.
#     """
#     query    = query_dict["question"]
#     keywords = [kw.lower() for kw in query_dict["keywords"]]

#     if pipeline == "enhanced":
#         reranked = enhanced_pipeline(query, retrieve_k=20,
#                                      final_k=5, strategy="both")
#         chunks = [r["chunk"]["text"].lower() for r in reranked]
#     else:
#         chunks = [c["text"].lower()
#                   for c in baseline_retrieve(query, k=5)]

#     # Find rank of first chunk containing any keyword
#     for rank, chunk_text in enumerate(chunks, 1):
#         if any(kw in chunk_text for kw in keywords):
#             return 1.0 / rank

#     return 0.0  # no relevant chunk found


# # ============================================================
# # GENERATION EVALUATION — Claude as judge
# # ============================================================

# def evaluate_answer(question, expected, actual_answer, source_chunks):
#     """
#     Uses Claude to score the generated answer on 4 dimensions:
#       Faithfulness   : answer stays within retrieved chunks (1-3)
#       Correctness    : answer matches expected answer (1-3)
#       Completeness   : answer covers the full question (1-3)
#       Grounding      : answer cites sources properly (1-3)

#     Returns dict of scores + brief justification.
#     Uses Claude as an automated judge — standard practice in
#     RAG evaluation (similar to RAGAS framework).
#     """
#     chunks_text = "\n\n".join([
#         f"[Chunk {i+1}]: {c['text'][:300]}"
#         for i, c in enumerate(source_chunks)
#     ])

#     judge_prompt = f"""You are evaluating a RAG system answer for a UK energy sector Q&A tool.

# Question: {question}

# Expected answer (reference): {expected}

# Retrieved chunks used:
# {chunks_text}

# Generated answer: {actual_answer}

# Score the generated answer on these 4 dimensions (1=poor, 2=partial, 3=good):

# 1. Faithfulness: Does the answer only use information from the retrieved chunks?
#    (3=fully grounded, 2=mostly grounded with minor inference, 1=hallucinated content)

# 2. Correctness: How accurate is the answer compared to the expected answer?
#    (3=fully correct, 2=partially correct, 1=wrong or missing key facts)

# 3. Completeness: Does the answer address the full question?
#    (3=fully complete, 2=partially complete, 1=incomplete or off-topic)

# 4. Grounding: Does the answer cite sources with [1][2] style references?
#    (3=clear citations, 2=implicit source references, 1=no citations)

# Respond ONLY in this exact JSON format:
# {{
#   "faithfulness": <1-3>,
#   "correctness": <1-3>,
#   "completeness": <1-3>,
#   "grounding": <1-3>,
#   "justification": "<one sentence explanation>"
# }}"""

#     response = client.messages.create(
#         model=GENERATION_MODEL,
#         max_tokens=200,
#         messages=[{"role": "user", "content": judge_prompt}]
#     )

#     try:
#         scores = json.loads(response.content[0].text.strip())
#     except json.JSONDecodeError:
#         # Fallback if JSON parsing fails
#         scores = {
#             "faithfulness" : 1,
#             "correctness"  : 1,
#             "completeness" : 1,
#             "grounding"    : 1,
#             "justification": "Parse error — manual review needed"
#         }
#     return scores


# # ============================================================
# # FULL EVALUATION RUN
# # ============================================================

# def run_evaluation(queries=EVAL_QUERIES):
#     """
#     Runs full evaluation on all queries:
#       1. Both pipelines generate answers
#       2. MRR computed for retrieval quality
#       3. Claude judges generation quality
#       4. Results stored + summary table printed
#     """
#     results = []

#     for q in queries:
#         print(f"\n{'='*60}")
#         print(f"Evaluating {q['id']} [{q['type']}]: {q['question'][:60]}...")
#         print(f"{'='*60}")

#         # ── Baseline ─────────────────────────────────────────
#         print("  Running baseline...")
#         b_result  = baseline_generate(q["question"])
#         b_mrr     = compute_mrr(q, pipeline="baseline")
#         b_scores  = evaluate_answer(
#             q["question"],
#             q["expected_answer"],
#             b_result["answer"],
#             b_result["chunks"]
#         )

#         # ── Enhanced ─────────────────────────────────────────
#         print("  Running enhanced...")
#         e_result  = enhanced_generate(q["question"], strategy="both")
#         e_mrr     = compute_mrr(q, pipeline="enhanced")
#         e_scores  = evaluate_answer(
#             q["question"],
#             q["expected_answer"],
#             e_result["answer"],
#             e_result["chunks"]
#         )

#         # ── Store result ──────────────────────────────────────
#         results.append({
#             "id"              : q["id"],
#             "type"            : q["type"],
#             "question"        : q["question"][:50] + "...",

#             # Retrieval
#             "baseline_mrr"    : round(b_mrr, 3),
#             "enhanced_mrr"    : round(e_mrr, 3),
#             "mrr_delta"       : round(e_mrr - b_mrr, 3),

#             # Generation scores — baseline
#             "b_faithful"      : b_scores["faithfulness"],
#             "b_correct"       : b_scores["correctness"],
#             "b_complete"      : b_scores["completeness"],
#             "b_grounding"     : b_scores["grounding"],
#             "b_total"         : sum([b_scores["faithfulness"],
#                                      b_scores["correctness"],
#                                      b_scores["completeness"],
#                                      b_scores["grounding"]]),

#             # Generation scores — enhanced
#             "e_faithful"      : e_scores["faithfulness"],
#             "e_correct"       : e_scores["correctness"],
#             "e_complete"      : e_scores["completeness"],
#             "e_grounding"     : e_scores["grounding"],
#             "e_total"         : sum([e_scores["faithfulness"],
#                                      e_scores["correctness"],
#                                      e_scores["completeness"],
#                                      e_scores["grounding"]]),

#             # Raw answers for demo log
#             "baseline_answer" : b_result["answer"],
#             "enhanced_answer" : e_result["answer"],
#             "b_justification" : b_scores.get("justification", ""),
#             "e_justification" : e_scores.get("justification", "")
#         })

#         print(f"  ✅ Baseline  — MRR: {b_mrr:.3f} | Score: {results[-1]['b_total']}/12")
#         print(f"  ✅ Enhanced  — MRR: {e_mrr:.3f} | Score: {results[-1]['e_total']}/12")

#     return results


# # ============================================================
# # SUMMARY TABLE + SAVE
# # ============================================================

# def print_summary(results):
#     """Prints a clean summary table and aggregate statistics."""

#     df = pd.DataFrame(results)

#     print("\n" + "=" * 70)
#     print("EVALUATION SUMMARY")
#     print("=" * 70)

#     # Retrieval summary
#     print("\n📊 RETRIEVAL — MRR (higher = better, max 1.0)")
#     print(f"  Baseline  avg MRR : {df['baseline_mrr'].mean():.3f}")
#     print(f"  Enhanced  avg MRR : {df['enhanced_mrr'].mean():.3f}")
#     print(f"  Improvement       : +{(df['enhanced_mrr'].mean() - df['baseline_mrr'].mean()):.3f}")

#     # Generation summary
#     print("\n📊 GENERATION — Total score (higher = better, max 12)")
#     print(f"  Baseline  avg score : {df['b_total'].mean():.1f}/12")
#     print(f"  Enhanced  avg score : {df['e_total'].mean():.1f}/12")
#     print(f"  Improvement         : +{(df['e_total'].mean() - df['b_total'].mean()):.1f}")

#     # Per query table
#     print("\n📋 PER QUERY BREAKDOWN")
#     display_cols = [
#         "id", "type",
#         "baseline_mrr", "enhanced_mrr",
#         "b_total", "e_total"
#     ]
#     print(df[display_cols].to_string(index=False))

#     # Per query type breakdown
#     print("\n📋 BY QUERY TYPE")
#     type_summary = df.groupby("type").agg(
#         baseline_mrr=("baseline_mrr", "mean"),
#         enhanced_mrr=("enhanced_mrr", "mean"),
#         baseline_score=("b_total", "mean"),
#         enhanced_score=("e_total", "mean")
#     ).round(2)
#     print(type_summary.to_string())

#     return df


# # ============================================================
# # INTERACTIVE: ADD YOUR OWN QUERY
# # ============================================================

# def test_single_query(question, expected_answer="Not specified",
#                       keywords=None, query_type="custom"):
#     """
#     Test a single custom query through both pipelines.
#     Use this to interactively test any question.

#     Usage:
#       test_single_query(
#           question        = "Your question here",
#           expected_answer = "What you expect the answer to be",
#           keywords        = ["key", "terms", "to", "check"],
#           query_type      = "custom"
#       )
#     """
#     if keywords is None:
#         keywords = question.lower().split()[:5]

#     custom_q = {
#         "id"              : "CUSTOM",
#         "type"            : query_type,
#         "question"        : question,
#         "expected_answer" : expected_answer,
#         "keywords"        : keywords
#     }

#     print(f"\n{'='*60}")
#     print(f"CUSTOM QUERY: {question}")
#     print(f"{'='*60}")

#     # Baseline
#     b_result = baseline_generate(question)
#     b_mrr    = compute_mrr(custom_q, pipeline="baseline")
#     print(f"\n📌 BASELINE (MRR: {b_mrr:.3f})")
#     print(b_result["answer"])

#     # Enhanced
#     e_result = enhanced_generate(question, strategy="both")
#     e_mrr    = compute_mrr(custom_q, pipeline="enhanced")
#     print(f"\n🚀 ENHANCED (MRR: {e_mrr:.3f})")
#     print(e_result["answer"])

#     # Score both
#     b_scores = evaluate_answer(question, expected_answer,
#                                b_result["answer"], b_result["chunks"])
#     e_scores = evaluate_answer(question, expected_answer,
#                                e_result["answer"], e_result["chunks"])

#     print(f"\n📊 SCORES (Faithful | Correct | Complete | Grounding | Total)")
#     print(f"  Baseline : {b_scores['faithfulness']} | {b_scores['correctness']} | "
#           f"{b_scores['completeness']} | {b_scores['grounding']} | "
#           f"{sum([b_scores['faithfulness'], b_scores['correctness'], b_scores['completeness'], b_scores['grounding']])}/12")
#     print(f"  Enhanced : {e_scores['faithfulness']} | {e_scores['correctness']} | "
#           f"{e_scores['completeness']} | {e_scores['grounding']} | "
#           f"{sum([e_scores['faithfulness'], e_scores['correctness'], e_scores['completeness'], e_scores['grounding']])}/12")
#     print(f"\n  Baseline justification: {b_scores['justification']}")
#     print(f"  Enhanced justification: {e_scores['justification']}")

#     return b_result, e_result, b_scores, e_scores


# # ============================================================
# # RUN FULL EVALUATION
# # ============================================================
# print("Starting full evaluation — this will take a few minutes...")
# print("Each query runs both pipelines + scoring = ~6 API calls per query")
# print(f"Total queries: {len(EVAL_QUERIES)} × 6 calls = ~{len(EVAL_QUERIES)*6} API calls\n")

# eval_results = run_evaluation(EVAL_QUERIES)
# df_results   = print_summary(eval_results)

# # Save results to CSV for report
# df_results.to_csv('/content/evaluation_results.csv', index=False)
# print("\n✅ Results saved to /content/evaluation_results.csv")

# # ============================================================
# # EXAMPLE: Test your own custom query
# # ============================================================
# # Uncomment and modify to test any query interactively:

# # test_single_query(
# #     question        = "What is the de-rated capacity margin for winter 2025/26?",
# #     expected_answer = "De-rated capacity margin is approximately 5.2 GW",
# #     keywords        = ["de-rated", "margin", "GW", "capacity"],
# #     query_type      = "custom"
# # )

Starting full evaluation — this will take a few minutes...
Each query runs both pipelines + scoring = ~6 API calls per query
Total queries: 10 × 6 calls = ~60 API calls


Evaluating Q01 [simple_fact]: What is the peak national demand forecast for winter 2025/26...
  Running baseline...
  Running enhanced...
  [Rewritten]: Peak national electricity demand forecast winter 2025/26 ACS (Annual Capacity Statement) transmission system requirements GB (Great Britain) maximum load MW (megawatts) cold spell scenarios NESO projections
  [HyDE]:      The peak national demand forecast for Winter 2025/26 is projected at 59.8 GW under Average Cold Spell (ACS) conditions, representing a modest increase...

  [Rewritten]: Peak national demand forecast winter 2025/26 ACS transmission system demand MW GW National Grid ESO NESO Annual Energy Review seasonal peak load electricity consumption projections underlying demand weather-corrected
  [HyDE]:      The peak national demand forecast for winter 2025/26

In [22]:
# ============================================================
# CELL 10 — Step 5: Evaluation Framework (Revised)
# ============================================================
# Fixes applied:
#   1. Tighter MRR keywords — specific figures/phrases, not common words
#   2. JSON parsing fix — strips markdown fences before parsing
#   3. Judge prompt updated — explicitly requests raw JSON
#   4. Reduced to 2 queries per type (8 total) — efficient API usage
#
# Query types covered:
#   simple_fact  (2) — exact figure retrieval
#   deep_context (2) — multi-chunk synthesis
#   ambiguous    (2) — broad query handling
#   edge_case    (2) — corpus boundary + hallucination resistance

import pandas as pd
import json

# ============================================================
# EVALUATION QUERY SET — 8 queries, 2 per type
# ============================================================

EVAL_QUERIES = [
    # ── Simple factual ────────────────────────────────────────
    {
        "id"              : "Q01",
        "type"            : "simple_fact",
        "question"        : "What is the peak national demand forecast for winter 2025/26?",
        "expected_answer" : "Central forecast peak demand 42,613 MW, upper bound 46,612 MW, minimum lower bound 22,016 MW",
        "keywords"        : ["42,613", "46,612", "22,016"]
    },
    {
        "id"              : "Q02",
        "type"            : "simple_fact",
        "question"        : "What is the capacity of the Greenlink interconnector?",
        "expected_answer" : "0.5 GW interconnector between Ireland and Great Britain commissioned in early 2025",
        "keywords"        : ["greenlink", "0.5 gw", "500 mw"]
    },

    # ── Deep context ──────────────────────────────────────────
    {
        "id"              : "Q03",
        "type"            : "deep_context",
        "question"        : "What tools does NESO use to balance the electricity system during winter?",
        "expected_answer" : "Balancing Mechanism, Demand Flexibility Service, reserve requirements, East Coast CMIS 900MW",
        "keywords"        : ["balancing mechanism", "demand flexibility service", "cmis"]
    },
    {
        "id"              : "Q04",
        "type"            : "deep_context",
        "question"        : "What are the future energy scenarios for GB electricity demand by 2050?",
        "expected_answer" : "System Transformation 705 TWh, Early Evolution 785 TWh, Headroom for Equity 797 TWh, Falling Behind 559 TWh",
        "keywords"        : ["705 twh", "785 twh", "559 twh"]
    },

    # ── Ambiguous ─────────────────────────────────────────────
    {
        "id"              : "Q05",
        "type"            : "ambiguous",
        "question"        : "How secure is UK energy this winter?",
        "expected_answer" : "Adequate margins, de-rated capacity margin increase of 0.3 GW from Greenlink, broadly similar to previous year",
        "keywords"        : ["de-rated capacity margin", "greenlink", "adequacy"]
    },
    {
        "id"              : "Q06",
        "type"            : "ambiguous",
        "question"        : "What is the energy market outlook?",
        "expected_answer" : "Ofgem state of market covers retail market conditions, wholesale prices, network charges and regulatory context",
        "keywords"        : ["ofgem", "wholesale", "retail"]
    },

    # ── Edge case ─────────────────────────────────────────────
    {
        "id"              : "Q07",
        "type"            : "edge_case",
        "question"        : "What happens if there is a gas supply disruption during winter 2025/26?",
        "expected_answer" : "Context may not fully cover this — tests corpus boundary on gas emergency procedures",
        "keywords"        : ["gas", "disruption", "contingency"]
    },
    {
        "id"              : "Q08",
        "type"            : "edge_case",
        "question"        : "What is the carbon intensity of the GB electricity grid in 2025?",
        "expected_answer" : "Context does not contain this — tests hallucination resistance",
        "keywords"        : ["carbon intensity", "gco2", "emissions"]
    },
]


# ============================================================
# RETRIEVAL EVALUATION — MRR
# ============================================================

def compute_mrr(query_dict, pipeline="enhanced"):
    """
    Mean Reciprocal Rank — measures how highly the first relevant
    chunk ranks. Uses specific keyword phrases, not common words,
    to avoid trivial matches.

    MRR = 1/rank of first relevant chunk
      Rank 1 = 1.0  |  Rank 2 = 0.5  |  Not found = 0.0
    """
    query    = query_dict["question"]
    keywords = [kw.lower() for kw in query_dict["keywords"]]

    if pipeline == "enhanced":
        reranked = enhanced_pipeline(
            query, retrieve_k=20, final_k=5, strategy="both"
        )
        chunks = [r["chunk"]["text"].lower() for r in reranked]
    else:
        chunks = [c["text"].lower()
                  for c in baseline_retrieve(query, k=5)]

    # First chunk containing ANY of the specific keywords
    for rank, chunk_text in enumerate(chunks, 1):
        if any(kw in chunk_text for kw in keywords):
            return 1.0 / rank

    return 0.0


# ============================================================
# GENERATION EVALUATION — Claude as judge
# ============================================================

def evaluate_answer(question, expected, actual_answer, source_chunks):
    """
    Claude scores the generated answer on 4 dimensions (1-3 each):
      Faithfulness  — grounded in retrieved chunks only
      Correctness   — matches expected answer
      Completeness  — addresses the full question
      Grounding     — cites sources with [1][2] references

    Fix: strips markdown fences before JSON parsing.
    Fix: judge prompt explicitly requests raw JSON.
    """
    chunks_text = "\n\n".join([
        f"[Chunk {i+1}]: {c['text'][:300]}"
        for i, c in enumerate(source_chunks)
    ])

    judge_prompt = f"""You are evaluating a RAG system answer for a UK energy sector Q&A tool.

Question: {question}

Expected answer (reference): {expected}

Retrieved chunks used:
{chunks_text}

Generated answer: {actual_answer}

Score the generated answer on these 4 dimensions (1=poor, 2=partial, 3=good):

1. Faithfulness: Does the answer only use information from the retrieved chunks?
   3=fully grounded in chunks | 2=mostly grounded, minor inference | 1=hallucinated content present

2. Correctness: How accurate is the answer vs the expected answer?
   3=fully correct with key facts | 2=partially correct | 1=wrong or missing key facts

3. Completeness: Does the answer fully address the question?
   3=fully complete | 2=partially addresses it | 1=incomplete or off-topic

4. Grounding: Does the answer cite sources using [1][2] style references?
   3=clear inline citations | 2=implicit source references | 1=no citations at all

IMPORTANT: Respond ONLY in raw JSON with no markdown, no code fences, no explanation.
Use exactly this format:
{{"faithfulness": <1-3>, "correctness": <1-3>, "completeness": <1-3>, "grounding": <1-3>, "justification": "<one sentence>"}}"""

    response = client.messages.create(
        model=GENERATION_MODEL,
        max_tokens=200,
        messages=[{"role": "user", "content": judge_prompt}]
    )

    # Fix: strip markdown fences before parsing
    raw = response.content[0].text.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()

    try:
        scores = json.loads(raw)
        # Validate expected keys exist
        for key in ["faithfulness", "correctness", "completeness", "grounding"]:
            if key not in scores:
                scores[key] = 1
        if "justification" not in scores:
            scores["justification"] = "No justification provided"
    except (json.JSONDecodeError, KeyError):
        scores = {
            "faithfulness" : 1,
            "correctness"  : 1,
            "completeness" : 1,
            "grounding"    : 1,
            "justification": "JSON parse error — manual review needed"
        }
    return scores


# ============================================================
# FULL EVALUATION RUN
# ============================================================

def run_evaluation(queries=EVAL_QUERIES):
    """
    Runs full evaluation on all queries:
      1. Both pipelines generate answers
      2. MRR computed for retrieval quality
      3. Claude judges generation quality on 4 dimensions
      4. Results stored + summary table printed
    """
    results = []

    for q in queries:
        print(f"\n{'='*60}")
        print(f"Evaluating {q['id']} [{q['type']}]: {q['question'][:55]}...")
        print(f"{'='*60}")

        # ── Baseline ─────────────────────────────────────────
        print("  Running baseline...")
        b_result = baseline_generate(q["question"])
        b_mrr    = compute_mrr(q, pipeline="baseline")
        b_scores = evaluate_answer(
            q["question"],
            q["expected_answer"],
            b_result["answer"],
            b_result["chunks"]
        )

        # ── Enhanced ─────────────────────────────────────────
        print("  Running enhanced...")
        e_result = enhanced_generate(q["question"], strategy="both")
        e_mrr    = compute_mrr(q, pipeline="enhanced")
        e_scores = evaluate_answer(
            q["question"],
            q["expected_answer"],
            e_result["answer"],
            e_result["chunks"]
        )

        # ── Print live result ─────────────────────────────────
        b_total = sum([b_scores["faithfulness"], b_scores["correctness"],
                       b_scores["completeness"], b_scores["grounding"]])
        e_total = sum([e_scores["faithfulness"], e_scores["correctness"],
                       e_scores["completeness"], e_scores["grounding"]])

        print(f"  ✅ Baseline — MRR: {b_mrr:.3f} | Score: {b_total}/12 | {b_scores['justification'][:60]}")
        print(f"  ✅ Enhanced — MRR: {e_mrr:.3f} | Score: {e_total}/12 | {e_scores['justification'][:60]}")

        results.append({
            "id"              : q["id"],
            "type"            : q["type"],
            "question"        : q["question"][:50] + "...",

            "baseline_mrr"    : round(b_mrr, 3),
            "enhanced_mrr"    : round(e_mrr, 3),
            "mrr_delta"       : round(e_mrr - b_mrr, 3),

            "b_faithful"      : b_scores["faithfulness"],
            "b_correct"       : b_scores["correctness"],
            "b_complete"      : b_scores["completeness"],
            "b_grounding"     : b_scores["grounding"],
            "b_total"         : b_total,

            "e_faithful"      : e_scores["faithfulness"],
            "e_correct"       : e_scores["correctness"],
            "e_complete"      : e_scores["completeness"],
            "e_grounding"     : e_scores["grounding"],
            "e_total"         : e_total,

            "baseline_answer" : b_result["answer"],
            "enhanced_answer" : e_result["answer"],
            "b_justification" : b_scores.get("justification", ""),
            "e_justification" : e_scores.get("justification", "")
        })

    return results


# ============================================================
# SUMMARY TABLE
# ============================================================

def print_summary(results):
    df = pd.DataFrame(results)

    print("\n" + "=" * 70)
    print("EVALUATION SUMMARY")
    print("=" * 70)

    print("\n📊 RETRIEVAL — MRR (higher = better, max 1.0)")
    print(f"  Baseline  avg MRR : {df['baseline_mrr'].mean():.3f}")
    print(f"  Enhanced  avg MRR : {df['enhanced_mrr'].mean():.3f}")
    print(f"  Improvement       : +{(df['enhanced_mrr'].mean() - df['baseline_mrr'].mean()):.3f}")

    print("\n📊 GENERATION — Total score (higher = better, max 12)")
    print(f"  Baseline  avg score : {df['b_total'].mean():.1f}/12")
    print(f"  Enhanced  avg score : {df['e_total'].mean():.1f}/12")
    print(f"  Improvement         : +{(df['e_total'].mean() - df['b_total'].mean()):.1f}")

    print("\n📋 PER QUERY BREAKDOWN")
    display_cols = ["id", "type", "baseline_mrr", "enhanced_mrr",
                    "b_total", "e_total"]
    print(df[display_cols].to_string(index=False))

    print("\n📋 BY QUERY TYPE")
    type_summary = df.groupby("type").agg(
        baseline_mrr    = ("baseline_mrr", "mean"),
        enhanced_mrr    = ("enhanced_mrr", "mean"),
        baseline_score  = ("b_total", "mean"),
        enhanced_score  = ("e_total", "mean")
    ).round(2)
    print(type_summary.to_string())

    print("\n📋 PER DIMENSION AVERAGES")
    print(f"  Baseline  — Faithful: {df['b_faithful'].mean():.1f} | "
          f"Correct: {df['b_correct'].mean():.1f} | "
          f"Complete: {df['b_complete'].mean():.1f} | "
          f"Grounding: {df['b_grounding'].mean():.1f}")
    print(f"  Enhanced  — Faithful: {df['e_faithful'].mean():.1f} | "
          f"Correct: {df['e_correct'].mean():.1f} | "
          f"Complete: {df['e_complete'].mean():.1f} | "
          f"Grounding: {df['e_grounding'].mean():.1f}")

    return df


# ============================================================
# INTERACTIVE: TEST A SINGLE CUSTOM QUERY
# ============================================================

def test_single_query(question, expected_answer="Not specified",
                      keywords=None, query_type="custom"):
    """
    Test any single query through both pipelines interactively.

    Usage:
      test_single_query(
          question        = "Your question here",
          expected_answer = "What you expect",
          keywords        = ["specific", "phrases", "to", "match"],
          query_type      = "custom"
      )
    """
    if keywords is None:
        keywords = question.lower().split()[:5]

    custom_q = {
        "id"              : "CUSTOM",
        "type"            : query_type,
        "question"        : question,
        "expected_answer" : expected_answer,
        "keywords"        : keywords
    }

    print(f"\n{'='*60}")
    print(f"CUSTOM QUERY: {question}")
    print(f"{'='*60}")

    b_result = baseline_generate(question)
    b_mrr    = compute_mrr(custom_q, pipeline="baseline")
    print(f"\n📌 BASELINE (MRR: {b_mrr:.3f})")
    print(b_result["answer"])

    e_result = enhanced_generate(question, strategy="both")
    e_mrr    = compute_mrr(custom_q, pipeline="enhanced")
    print(f"\n🚀 ENHANCED (MRR: {e_mrr:.3f})")
    print(e_result["answer"])

    b_scores = evaluate_answer(question, expected_answer,
                               b_result["answer"], b_result["chunks"])
    e_scores = evaluate_answer(question, expected_answer,
                               e_result["answer"], e_result["chunks"])

    b_total = sum([b_scores["faithfulness"], b_scores["correctness"],
                   b_scores["completeness"], b_scores["grounding"]])
    e_total = sum([e_scores["faithfulness"], e_scores["correctness"],
                   e_scores["completeness"], e_scores["grounding"]])

    print(f"\n📊 SCORES  (Faithful | Correct | Complete | Grounding | Total)")
    print(f"  Baseline : {b_scores['faithfulness']} | {b_scores['correctness']} | "
          f"{b_scores['completeness']} | {b_scores['grounding']} | {b_total}/12")
    print(f"  Enhanced : {e_scores['faithfulness']} | {e_scores['correctness']} | "
          f"{e_scores['completeness']} | {e_scores['grounding']} | {e_total}/12")
    print(f"\n  Baseline: {b_scores['justification']}")
    print(f"  Enhanced: {e_scores['justification']}")

    return b_result, e_result, b_scores, e_scores


# ============================================================
# RUN FULL EVALUATION
# ============================================================
print("Starting evaluation — 8 queries × ~8 API calls = ~64 calls")
print("Estimated cost: ~$0.03-0.05\n")

eval_results = run_evaluation(EVAL_QUERIES)
df_results   = print_summary(eval_results)

df_results.to_csv('/content/evaluation_results.csv', index=False)
print("\n✅ Results saved to /content/evaluation_results.csv")

# ============================================================
# EXAMPLE: Test your own custom query
# ============================================================
# Uncomment to test interactively:

# test_single_query(
#     question        = "What is the de-rated capacity margin for winter 2025/26?",
#     expected_answer = "De-rated capacity margin increased by 0.3 GW due to Greenlink",
#     keywords        = ["de-rated capacity margin", "0.3 gw", "greenlink"],
#     query_type      = "custom"
# )

Starting evaluation — 8 queries × ~8 API calls = ~64 calls
Estimated cost: ~$0.03-0.05


Evaluating Q01 [simple_fact]: What is the peak national demand forecast for winter 20...
  Running baseline...
  Running enhanced...
  [Rewritten]: Peak national electricity demand forecast winter 2025/26 ACS (Annual Cold Spell) transmission system demand MW GW NESO FES (Future Energy Scenarios) winter outlook capacity adequacy
  [HyDE]:      The peak national demand forecast for winter 2025/26 is projected at 61.4 GW under Average Cold Spell (ACS) conditions, representing a modest increase...

  [Rewritten]: Peak national electricity demand forecast winter 2025/26 ACS (Annual Connections Statement) ETYS (Electricity Ten Year Statement) transmission system peak load MW GW capacity margin adequacy
  [HyDE]:      The peak national demand forecast for winter 2025/26 is projected at 61.4 GW under Average Cold Spell (ACS) conditions, representing a 1.8% increase f...

  ✅ Baseline — MRR: 0.500 | Score: 